# Attention 메커니즘 완전 정복 - 실습 코드 1: Self-Attention 구현 (PyTorch)

- Tutorial ID: `expand-attention-mechanism`
- Tutorial: Attention 메커니즘 완전 정복
- Section ID: `expand-attention-mechanism-code-1`
- Section: 실습 코드 1: Self-Attention 구현 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Self-Attention 구현 (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
        def __init__(self, d_model, d_k):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_k)
        self.scale = d_k ** 0.5

        def forward(self, x, mask=None):
                Q = self.W_q(x)  # (batch, seq, d_k)
                K = self.W_k(x)  # (batch, seq, d_k)
                V = self.W_v(x)  # (batch, seq, d_k)

        # Attention scores
                scores = torch.bmm(Q, K.transpose(1,2)) / self.scale

        if mask is not None:
                        scores = scores.masked_fill(mask == 0, -1e9)

                weights = F.softmax(scores, dim=-1)
                output = torch.bmm(weights, V)
        return output, weights

# 사용 예시
attn = SelfAttention(d_model=512, d_k=64)
x = torch.randn(2, 10, 512)  # batch=2, seq=10, d=512
out, weights = attn(x)
print(f"Output: {out.shape}, Weights: {weights.shape}")